In [ ]:
import pandas as pd
import random
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os
from PIL import Image
import copy
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models, get_model_weights
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

import torch.optim as optim

import cv2 
import copy
import time
%matplotlib inline

In [ ]:
def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"[INFO] Seed: {seed}")

set_seed(42)

# Load data

In [3]:
df = pd.read_csv('/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')
df

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
...,...,...,...,...,...,...,...
10010,HAM_0002867,ISIC_0033084,akiec,histo,40.0,male,abdomen
10011,HAM_0002867,ISIC_0033550,akiec,histo,40.0,male,abdomen
10012,HAM_0002867,ISIC_0033536,akiec,histo,40.0,male,abdomen
10013,HAM_0000239,ISIC_0032854,akiec,histo,80.0,male,face


In [4]:
le = LabelEncoder()
dx_encoded = le.fit_transform(df['dx'])
df['dx_encoded'] = dx_encoded

# EDA

**Only images will be used for classification.**

In [5]:
df['dx_encoded'].value_counts()

dx_encoded
5    6705
4    1113
2    1099
1     514
0     327
6     142
3     115
Name: count, dtype: int64

**We have a class imbalance**

Plan
1. Resnet(18/152),   
   DenseNet(121/161),   
   ConvNeXt(tiny/large),   
   RegNet(y_400mf/x_32gf),   
   MobileNetV3(small/large),   
   ShuffleNetV2(x0_5/x2_0),   
   EfficientNet(b0/b7),   
   EfficientNetV2(s/l),   
   VisionTransformer(b_16/h_14),   
   SwinTransformer(t/v2_b), DeiT/ConvNeXtV2/CaiT (after) with gradual unfreeze\learning rate decay without oversampling\weighted loss
2. add weighted loss
3. add oversampling
4. Compare training\inference time, accuracy
5. add table feature

ResNeXt101_64, 

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df[['image_id',	'age',	'sex',	'localization']], 
    df['dx_encoded'],
    test_size=0.2,
    random_state=42,
    stratify=df['dx']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, 
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

In [7]:
part_1 = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
part_2 = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2'

part_1_files = [os.path.join(part_1, f) for f in os.listdir(part_1) if os.path.getsize(os.path.join(part_1, f)) > 0 and f.endswith('.jpg')]
part_2_files = [os.path.join(part_2, f) for f in os.listdir(part_2) if os.path.getsize(os.path.join(part_2, f)) > 0 and f.endswith('.jpg')]

all_files = part_1_files + part_2_files

In [8]:
df_all_files_img = pd.DataFrame({'dir_img':all_files})
df_all_files_img['image_id'] = df_all_files_img['dir_img'].apply(lambda x: x.split('/')[5].split('.')[0])
df_all_files_img

,dir_img,image_id
0,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0028933
1,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0028394
2,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0027799
3,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0028100
4,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0027960
...,...,...
10010,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0029733
10011,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0033470
10012,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0032153
10013,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0030216


In [9]:
X_train_img = list(X_train.merge(df_all_files_img,
                            how='inner',
                            on='image_id')['dir_img'])

X_val_img = list(X_val.merge(df_all_files_img,
                            how='inner',
                            on='image_id')['dir_img'])

X_test_img = list(X_test.merge(df_all_files_img,
                            how='inner',
                            on='image_id')['dir_img'])

# Function for auto training

In [10]:
#Create dataset
class SkinDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert('RGB')
        
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [ ]:
def build_model_and_transforms(
    model_name: str,
    num_classes: int,
    pretrained: bool = True,
    strong_aug: bool = True,
    device: str = "cpu"
):
    model_name = model_name.lower()

    # --- 1. Load weights and metadata ---
    try:
        weights_enum = get_model_weights(model_name)
        weights = weights_enum.DEFAULT if pretrained else None
    except Exception as e:
        print(f"[WARN] Unable to get weights for {model_name}: {e}")
        weights = None

    # --- 2. Create model ---
    model = get_model(model_name, weights=weights)

    # --- 3. Change fc ---
    if hasattr(model, "fc"):  # ResNet, DenseNet
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif hasattr(model, "classifier"):  # EfficientNet, ConvNeXt, MobileNet
        if isinstance(model.classifier, nn.Sequential):
            in_features = model.classifier[-1].in_features
            model.classifier[-1] = nn.Linear(in_features, num_classes)
        else:
            in_features = model.classifier.in_features
            model.classifier = nn.Linear(in_features, num_classes)
    elif hasattr(model, "head"):  # ViT, Swin
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f"Неизвестная архитектура: {model_name}")

    # --- 4. Obtaining official transforms from scales ---
    if weights is not None:
        base_transforms = weights.transforms()
        mean, std = base_transforms.mean, base_transforms.std
        interpolation = base_transforms.interpolation
        crop_size = base_transforms.crop_size[0]
        resize_size = base_transforms.resize_size[0]
    else:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        interpolation = transforms.InterpolationMode.BICUBIC
        crop_size, resize_size = 224, 236

    # --- 5. Create train and val transforms ---
    train_tfms = [
        transforms.RandomResizedCrop(crop_size, scale=(0.8, 1.0),
                                     ratio=(0.75, 1.33),
                                     interpolation=interpolation),
        transforms.RandomHorizontalFlip()
    ]
    if strong_aug:
        train_tfms.append(
            transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)], p=0.8)
        )
    train_tfms.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
        transforms.RandomErasing(p=0.25)
    ])
    train_transform = transforms.Compose(train_tfms)

    val_test_transform = weights.transforms() if weights is not None else transforms.Compose([
        transforms.Resize(resize_size, interpolation=interpolation),
        transforms.CenterCrop(crop_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    model = model.to(device)
    return model, train_transform, val_test_transform

In [ ]:
def freeze_backbone_unfreeze_head(model):
    # All freeze
    for p in model.parameters():
        p.requires_grad = False

    # unfreeze head / fc / classifier
    if hasattr(model, "fc"):
        for p in model.fc.parameters():
            p.requires_grad = True
    elif hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif hasattr(model, "head"):
        for p in model.head.parameters():
            p.requires_grad = True

    return model

In [ ]:
def gradual_unfreeze(model, model_type=None, epoch=0, optimizer=None, every=5, lr=1e-4):
    """
    A universal gradual unfreeze for popular TorchVision architectures.
    Unfreezes layer by layer every epoch.
    """
    def add_to_optimizer(params):
        if optimizer is not None:
            optimizer.add_param_group({"params": params, "lr": lr})

    # === We define blocks by architecture ===
    groups = []

    if hasattr(model, "layer4"):  # ResNet
        groups = [model.layer1, model.layer2, model.layer3, model.layer4]
    elif hasattr(model, "features"):  # EfficientNet / ConvNeXt / MobileNet
        groups = list(model.features)
    elif hasattr(model, "stages"):  # Swin / ConvNeXt v2
        groups = list(model.stages)
    elif hasattr(model, "encoder") and hasattr(model.encoder, "layers"):  # ViT
        groups = list(model.encoder.layers)
    elif hasattr(model, "blocks"):  # RegNet / MLP-style / DeiT
        groups = list(model.blocks)
    else:
        print("[WARN] Unable to determine groups for gradual unfreeze.")
        return

    step = epoch // every
    if step == 0:
        return 

    # === Unfreeze blocks from the end ===
    n_to_unfreeze = min(step, len(groups))
    for block in groups[-n_to_unfreeze:]:
        for p in block.parameters():
            if not p.requires_grad:
                p.requires_grad = True
        add_to_optimizer(block.parameters())

    print(f"[INFO] Gradual unfreeze: unfreeze {n_to_unfreeze}/{len(groups)} blocks.")


In [13]:
def train_model(model, train_loader, val_loader, num_epochs, criterion, optimizer, 
                patience=5, unfreeze_fn=None, model_type=None, 
                device="cpu", scheduler=None, name_for_save='best_model'):
    train_losses, train_accuracies = [], []
    val_losses, val_accuracies = [], []

    best_val_accuracy = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    iter_without_improvements = 0

    model.to(device)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # If need unfreeze
        if unfreeze_fn is not None:
            unfreeze_fn(model, model_type, epoch, optimizer)

        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc="Train"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss /= total
        train_accuracy = correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Val"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)

        val_loss /= total
        val_accuracy = correct / total
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        print(f"Train loss: {train_loss:.4f}, acc: {train_accuracy:.4f} | "
              f"Val loss: {val_loss:.4f}, acc: {val_accuracy:.4f}")

        # === Scheduler ===
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # === Early stopping ===
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            iter_without_improvements = 0
            best_model_wts = copy.deepcopy(model.state_dict())
        else:
            iter_without_improvements += 1
            if iter_without_improvements > patience:
                print("Early stopping")
                break

    model.load_state_dict(best_model_wts)
    torch.save(model, f"{name_for_save}.pth")
    return model, train_losses, train_accuracies, val_losses, val_accuracies

# Training

In [ ]:
def get_training_config(architecture: str):
    """
    Returns optimal hyperparameters for the architecture:
    - batch_size
    - learning rate
    - num_epochs
    - lr_decay_patience
    """
    arch = architecture.lower()

    if any(x in arch for x in ["resnet18", "mobilenet", "shufflenet", "regnet_y_400mf"]):
        return dict(batch_size=64, lr=1e-3, num_epochs=20, patience=3)
    
    elif any(x in arch for x in ["resnet152", "densenet", "convnext_tiny", "swin_t", "efficientnet_b0", "efficientnetv2_s"]):
        return dict(batch_size=32, lr=1e-3, num_epochs=25, patience=3)
    
    elif any(x in arch for x in ["efficientnet_b7", "swin_b", "vit_b_16", "efficientnetv2_m"]):
        return dict(batch_size=16, lr=5e-4, num_epochs=25, patience=4)
    
    elif any(x in arch for x in ["efficientnetv2_l", "vit_h_14", "convnext_large"]):
        return dict(batch_size=8, lr=1e-4, num_epochs=30, patience=5)
    
    else:
        return dict(batch_size=32, lr=1e-3, num_epochs=25, patience=3)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = 7
criterion = nn.CrossEntropyLoss()

models_acr = [
    "resnet18",
    "efficientnet_b0",
    "efficientnetv2_l",
    "vit_b_16",
    "swin_t",
]


results = []

for architecture in models_acr:
    print(f"\n{'='*70}")
    print(f"🚀 Обучение модели: {architecture}")
    print(f"{'='*70}")

    # --- 1. Config for model ---
    cfg = get_training_config(architecture)
    print(f"[CONFIG] batch={cfg['batch_size']} | lr={cfg['lr']} | epochs={cfg['num_epochs']}")

    # --- 2. model and transformation ---
    model, train_tfms, val_tfms = build_model_and_transforms(
        architecture,
        num_classes=num_classes,
        pretrained=True,
        strong_aug=True,
        device=device
    )

    model = freeze_backbone_unfreeze_head(model)

    # --- 3. Datasets ans DataLoaders ---
    train_dataset = SkinDataset(train_files, train_labels, transform=train_tfms)
    val_dataset   = SkinDataset(val_files, val_labels, transform=val_tfms)

    train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=4)

    # --- 4. optimizer and scheduler ---
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr"],
        weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg["patience"]
    )

    # --- 5. Training ---
    model, train_losses, train_accs, val_losses, val_accs = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=cfg["num_epochs"],
        criterion=criterion,
        optimizer=optimizer,
        patience=5,
        unfreeze_fn=gradual_unfreeze,
        model_type=architecture,
        device=device,
        scheduler=scheduler,
        name_for_save=f"best_{architecture}"
    )

    best_val_acc = max(val_accs)
    results.append({"model": architecture, "best_val_acc": best_val_acc})
    print(f"✅ {architecture}: best val acc = {best_val_acc:.4f}")

# --- 6. Сравнение всех моделей ---
import pandas as pd
results_df = pd.DataFrame(results).sort_values(by="best_val_acc", ascending=False)
print("\n🏁 Итоговое сравнение:")
print(results_df)


In [ ]:
# Model prediction result
def evaluate_model(model, dataloader, device, class_names=None):
    model.eval()
    y_true, y_pred = [], []
    total_time, num_batches = 0.0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.time()
            outputs = model(images)
            if device.type == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            total_time += (end - start)
            num_batches += 1

            _, predicted = torch.max(outputs.data, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    # --- Метрики
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "avg_batch_time": total_time / num_batches,
        "avg_img_time": (total_time / num_batches) / dataloader.batch_size
    }

    print(f"\n🕒 Avg time per batch: {metrics['avg_batch_time']:.4f} sec")
    print(f"🖼️ Avg time per image: {metrics['avg_img_time']:.4f} sec")

    return metrics, cm, report

# Visualisation confusion matrix
def plot_confusion_matrix(cm, classes):
    with plt.style.context('default'):  
        plt.figure(figsize=(5, 4))
        sns.set(font_scale=1.0)
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False,
                    xticklabels=classes, yticklabels=classes)
        plt.xlabel('Predicted labels')
        plt.ylabel('True labels')
        plt.title('Confusion Matrix')
        plt.show()

In [ ]:
model_resnet18_base = torch.load("/kaggle/input/skin_resnet_18_base/pytorch/default/1/model_resnet18_base.pth",
                   map_location="cpu",
                   weights_only=False)
model_resnet18_base.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _ = model_resnet18_base(images)
        break

metrics, cm, report = evaluate_model(model_resnet18_base, test_loader,device=device)
print("Metrics for current model:")
print(pd.DataFrame([metrics]))
print(report)
plot_confusion_matrix(cm, classes=list(range(num_classes)))